# 08. 공정불량 예측 v2 (초기허용불량 제외)

**v1 병목**
- CN7 불량 15개 중 14개가 R1 → LORO에서 R1을 빼면 학습 불량 1개 → 지도학습 AP ≈ 기본 불량률(0.005)
- fold마다 점수 척도가 다른데 pooled AP 계산 → 지도학습 성능이 실제보다 나쁘게(또는 이상하게) 나옴
- 이상탐지 정상 기준이 labeled 양품 1천여 샷뿐

**v2 전략**
1. 정상 기준 = unlabeled 생산 샷 수만 개 (labeled 샷·Run 시간창 제거 → 누수 없음)
2. 지도학습 = CN7+RG3 통합 LORO (수준 무관 변수) → 학습 불량 최대 46개
3. 점수 = 정상 생산 샷 대비 척도 (robust z / 백분위) → fold·제품 간 비교·앙상블·임계값 일관
4. 메인 후보는 미리 고정(E2), 나머지는 ablation으로만 보고 → 6개 Run에서 30개 방법 중 최고를 고르는 과적합 방지

**v1 기준선 (loro_noinit, 비교용)**: CN7 T0+state·iforest AP 0.196 (미성형 계열 0.689, R@10% 0.27) / RG3 C5·logit AP 0.150 (R@10% 0.125)

In [ ]:
import os, json
import numpy as np
import pandas as pd
from scope import apply_scope
from model_v2 import (build_reference, reference_features, run_unsupervised, run_pooled_supervised,
                      ensemble, mask_transient, evaluate_v2, f1_v2, FSETS)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
OUT = './outputs/v2_noinit'
os.makedirs(OUT, exist_ok=True)
PRODUCTS = ['cn7', 'rg3']
UNL = './linked/linked_unlabeled_{}.csv'
REF_PERIOD = None            # 예: ('2020-09-01', '2020-12-31') → 대회 기간 근처만 정상 기준으로 (드리프트 비교용)
SCALES = json.load(open('./base/reference_scales.json', encoding='utf-8'))

FEAT, SHOTS = {}, {}
for p in PRODUCTS:
    SHOTS[p] = pd.read_csv(f'./base/{p}_shots.csv', parse_dates=['TimeStamp'])      # 제외 split 포함 전체 (reference 제거용)
    FEAT[p] = apply_scope(pd.read_csv(f'./base/{p}_features.csv', parse_dates=['TimeStamp']), True, False)
    FEAT[p]['Reason'] = FEAT[p]['Reason'].fillna('정상')
    d = FEAT[p]
    print(p.upper(), '샷', len(d), '불량', int(d['PassOrFail'].sum()), d.loc[d['PassOrFail'] == 1, 'Reason'].value_counts().to_dict())

## 1. reference (unlabeled 정상 생산 샷) + 같은 파생변수

In [ ]:
REF = {}
for p in PRODUCTS:
    cache = f'./base/ref_{p}_features{"" if REF_PERIOD is None else "_period"}.csv'
    if os.path.exists(cache):
        REF[p] = pd.read_csv(cache, parse_dates=['TimeStamp'])
    else:
        rs = build_reference(UNL.format(p), SHOTS[p], p, period=REF_PERIOD)
        REF[p] = reference_features(rs, SCALES[p]['steps'], SCALES[p]['fr_step'])
        REF[p].to_csv(cache, index=False, encoding='utf-8-sig')
    print(p, REF[p].shape)

# 점검: 수준 무관 변수(dev_*)는 labeled 양품과 reference 분포가 비슷해야 정상 기준으로 쓸 수 있음
chk = ['dev_Max_Injection_Speed', 'dev_Filling_Time', 'dev_Max_Injection_Pressure', 'dev_Cushion_Position',
       'Max_Injection_Speed_raw', 'Mold_Temperature_3_raw']
for p in PRODUCTS:
    good = FEAT[p][FEAT[p]['PassOrFail'] == 0]
    print(f'\n[{p}] 중앙값 / IQR  (labeled 양품 vs reference)')
    display(pd.DataFrame({'lab_med': good[chk].median(), 'ref_med': REF[p][chk].median(),
                          'lab_iqr': good[chk].quantile(.75) - good[chk].quantile(.25),
                          'ref_iqr': REF[p][chk].quantile(.75) - REF[p][chk].quantile(.25)}).round(3))

## 2. 이상탐지 (reference로만 학습 → fold 불필요)

In [ ]:
OOF, REF_Z = [], {}
for p in PRODUCTS:
    o, rz = run_unsupervised(FEAT[p], REF[p], p)
    OOF.append(o); REF_Z[p] = rz
OOF_U = pd.concat(OOF, ignore_index=True)

## 3. 통합 지도학습 (CN7+RG3 LORO)

In [ ]:
OOF_S, SUP_REF = run_pooled_supervised(FEAT, REF)

## 4. 앙상블 (멤버는 결과 보기 전에 고정)
- **E1** 이상탐지 3종 (free)
- **E2 (메인)** E1 + 통합 logit → 이벤트형(미성형)은 이상탐지, 학습 가능한 패턴은 지도학습이 보완
- **E3** E2 + 수준 변수 iforest → CN7 가스(Run 전체 금형온도↑) 같은 상태형 신호 보강 시도

In [ ]:
E1 = ['U_iforest|free', 'U_maha|free', 'U_knn|free']
E2 = E1 + ['S_logit|sup_free']
E3 = E2 + ['U_iforest|free+level']
ALL = pd.concat([OOF_U, OOF_S], ignore_index=True)
ENS = pd.concat([ensemble(ALL, m, n, REF_Z, SUP_REF) for m, n in [(E1, 'E1_U3'), (E2, 'E2_U3+logit'), (E3, 'E3_E2+level')]],
                ignore_index=True)
ALL = pd.concat([ALL, ENS], ignore_index=True)
ALL = pd.concat([ALL, mask_transient(ALL[ALL['method'].str.startswith('E')])], ignore_index=True)
ALL.to_csv(f'{OUT}/oof.csv', index=False, encoding='utf-8-sig')
print(ALL['method'].nunique(), 'methods')

## 5. 평가

In [ ]:
SUM = evaluate_v2(ALL)
SUM.to_csv(f'{OUT}/summary.csv', index=False, encoding='utf-8-sig')
cols = ['method', 'n_def', 'AP', 'AP_lo', 'AP_hi', 'AP/base', 'hit@5%', 'hit@10%', 'AP_가스', 'hit10_가스', 'AP_미성형', 'hit10_미성형']
for p in PRODUCTS:
    print(f'\n===== {p.upper()} =====')
    display(SUM[SUM['product'] == p][[c for c in cols if c in SUM]].round(3).head(25))

# 메인(E2) vs v1 기준선
V1 = {'cn7': 0.196, 'rg3': 0.150}
main = SUM[SUM['method'] == 'E2_U3+logit'].set_index('product')
display(pd.DataFrame({'v1 AP': V1, 'v2 E2 AP': main['AP'], 'v2 E2 CI': main['AP_lo'].round(3).astype(str) + '~' + main['AP_hi'].round(3).astype(str),
                      'v2 hit@10%': main['hit@10%']}))

## 6. F1 (임계값은 테스트 Run을 보지 않고 결정)

In [ ]:
SHOW = ['E1_U3', 'E2_U3+logit', 'E3_E2+level', 'E2_U3+logit|과도제외', 'S_logit|sup_free', 'U_iforest|free']
F1, PRED = f1_v2(ALL, methods=SHOW)
F1.to_csv(f'{OUT}/f1.csv', index=False, encoding='utf-8-sig')
display(F1.round(3))

# 메인 모델: 유형별 탐지 / Run별 오탐
m = PRED[PRED['method'] == 'E2_U3+logit']
display(m[m['y'] == 1].groupby(['product', 'Reason'])['pred'].agg(탐지='sum', 전체='size'))
display(m.groupby(['product', 'run_id']).apply(lambda g: pd.Series({
    '샷': len(g), '불량': int(g['y'].sum()), 'TP': int(((g['pred'] == 1) & (g['y'] == 1)).sum()),
    'FP': int(((g['pred'] == 1) & (g['y'] == 0)).sum())}), include_groups=False))

## 7. 체크리스트 (결과 해석)
- reference 점검표에서 dev_* 중앙값·IQR이 labeled 양품과 크게 다르면 → `REF_PERIOD`로 기간 제한해서 다시
- E2가 v1보다 CN7 미성형 계열 AP·RG3 AP 둘 다 올랐는지 / CI가 겹치는지 (Run 6개라 CI 넓음 → '개선 경향'으로 표현)
- `|과도제외`에서 잃는 실제 불량 수 확인 (RG3 R4 첫 불량이 3번째 샷)
- CN7 가스는 샷 단위 신호가 없음(Run 전체 특성) → 어떤 방법도 Run 내 순위로는 못 잡으면 보고서 '실패 조건'으로